In [ ]:
%%capture
%run script_nettoyage.ipynb

Analyse 1 : Evolution des fréquentations des musées par type de musées.

In [ ]:
from fonctions import tracer_comparaison_frequentation 

mes_donnees = {
    "Totale": df_freq_totale,
    "Gratuite": df_freq_gratuite,
    "Payante": df_freq_payante    
}

tracer_comparaison_frequentation(
    mes_donnees, 
    an='annee',           
    freq='freq_net'    
)

Nous souhaitons connaitre le nombre de musées par région

In [ ]:
from cartiflette import carti_download
from fonctions import nettoyer_texte

def generer_carte_musees(df_donnees):
    """
    Prend en entrée un DataFrame contenant les musées et affiche 
    directement la carte choroplèthe par région.
    """
    # Agrégation des données
    nb_musees_reg = (
        df_donnees
        .groupby("NOMREG", as_index=False)["REF DU MUSEE"]
        .nunique()
    )
    nb_musees_reg = nb_musees_reg.rename(columns={"REF DU MUSEE": "nb_musees"})

    # Téléchargement du fond de carte
    regions = carti_download(
        values=["France"],
        crs=4326,
        borders="REGION",
        vectorfile_format="geojson",
        simplification=50,
        filter_by="FRANCE_ENTIERE_DROM_RAPPROCHES",
        source="EXPRESS-COG-CARTO-TERRITOIRE",
        year=2022
    )
    # Nettoyage des textes et jointure

    nb_musees_reg["REGION_CLEAN"] = nettoyer_texte(nb_musees_reg['NOMREG'])
    regions["REGION_CLEAN"] = nettoyer_texte(regions["LIBELLE_REGION"])
    
    carte = regions.merge(nb_musees_reg, on="REGION_CLEAN", how="left")
    carte['nb_musees'] = carte['nb_musees'].fillna(0)

    # Création et affichage de la carte
    fig, ax = plt.subplots(1, 1, figsize=(12, 10)) 

    carte.plot(
        column="nb_musees",       
        ax=ax,
        cmap="OrRd",              
        edgecolor="black",        
        linewidth=0.8,            
        legend=True,              
        legend_kwds={
            'label': "Nombre de musées", 
            'orientation': "vertical", 
            'shrink': 0.7         
        }
    )

    plt.title("Répartition des musées par région", fontsize=16, fontweight='bold')
    plt.axis("off") 
    
    # Affichage final
    plt.show()

generer_carte_musees(df_freq_totale)

Nous souhaitons à présent représenter sur une carte de la France métropolitaine la fréquentation des musées par région. Cependant, nous voulons également représenter la fréquentation par rapport au nombre d'habitants de le région. Pour cela nous utilisons une table présentant le nombre d'habitants par région en 2010 (fichier data.py). Nous faisons ensuite une jointure de cette table avec notre table fréquentation_totale. Le fait de prendre comme 2010 comme référence pour toutes les années de notre base de données n'est pas précis; l'objectif n'est pas d'être précis mais de montrer les tendances. Nous créons la colonne freq_pour_1000_hab représentant la fréquentation du musée, une année donnée pour 1 000 habitants.

In [ ]:
from data import pop_2010

df_freq_totale = df_freq_totale.merge(pop_2010, on="NOMREG", how="left")

df_freq_totale["freq_pour_1000_hab"] = (
    pd.to_numeric(df_freq_totale["freq_net"], errors="coerce")
    / df_freq_totale["population_2010"] * 1000
)

df_freq_totale

La fonction carto_frequentation_region permet de représenter sur une carte de la France métropolitaine la fréquentation par région une année donnée par quartiles. L'argument col_freq permet de choisir entre représenter la fréquentation ou la fréquentation pour 1 000 habitants.

In [ ]:
from fonctions import carto_frequentation_region

carte_2022 = carto_frequentation_region(df_freq_totale, "2014", col_freq="freq_pour_1000_hab")

Au moins une région dépasse les 1 000 visiteurs pour 1 000 habitants. Qui dépasse ce seuil et en quelles années ?

In [ ]:
freq_region_annee = (
    df_freq_totale
    .groupby(["annee", "NOMREG"], as_index=False)["freq_pour_1000_hab"]
    .sum()
)

freq_region_annee.loc[
    freq_region_annee["freq_pour_1000_hab"] > 1000,
    ["annee", "NOMREG", "freq_pour_1000_hab"]
].sort_values(["NOMREG", "annee"])

La région Ile-De-France dépasse ce seuil tous les ans. Cela s'explique par le tourisme très important que connaît cette région.